# Apex model: T1-only prediction of incident diabetes at T2 and T3

Two models, same T1 feature definition, different horizons, each with its **own pre-split train/test file pair**:
- **Model A (T1 → T2):** predict `HBAC_T2 > 6.5` from T1 features. Files: `TRAIN_PATH_A` / `TEST_PATH_A`.
- **Model B (T1 → T3):** predict `HBAC_T3 > 6.5` from T1 features. Files: `TRAIN_PATH_B` / `TEST_PATH_B`. No T2 data is used anywhere in this model — not as a feature, not for cohort filtering.

**Cohort (both models):** drop missing `HBAC_T1`, exclude prevalent diabetes at baseline (`DIABETES_T1 == 1`) so the task is genuinely incident-case prediction rather than re-detecting existing diabetes, then drop rows missing the respective target flag. Applied independently to each of the 4 files.

**Features:** every `*_T1` column as-is, including its `*_T1_MISSING` companions and `HBAC_T1` as a continuous predictor — no further curation, since the incoming files are expected to already be preprocessed. `DIABETES_T1` is explicitly excluded (structurally constant within the cohort by construction — see the feature-selection section), and a zero-variance safety net catches anything else like it. Static/no-suffix baseline columns (`GENDER`, `SMOKING`, family history, `NSES`, ...) are **excluded by default** under this literal scope — flip `INCLUDE_STATIC_BASELINE_FEATURES` below if you want them back in; family-history variables in particular are a classical diabetes risk factor worth reconsidering.

**Dtype handling:** each pipeline's preprocessing step is a `ColumnTransformer` built from the actual dtypes of `X` — numeric columns are scaled (logistic regression) or passed through (tree models), any non-numeric column is one-hot encoded automatically. So mixed-type `*_T1` columns would be handled without manual configuration; today's real data happens to be all-numeric, so the one-hot branch is present but not yet exercised against real string/categorical data.

**Models:** Logistic Regression + ElasticNet, Random Forest, XGBoost — each wrapped in an `imblearn` pipeline with SMOTE applied only inside the training fold.

**Validation:** one Optuna Bayesian search (TPE sampler + median pruner for speed, 5-fold inner CV) per model/algorithm, tuned on AUPRC, refit on the full train file, then evaluated **once** on that model's real, never-touched test file. This is the actual unbiased performance estimate — there's no separate internal nested-CV layer, since a genuine external test set already provides the honest generalization check.

**Threshold:** chosen via inner CV on train data only (never touches the test file) to maximize F1.

**Metrics:** AUROC, AUPRC (threshold-free), precision/recall at the F1-optimal threshold, all with bootstrap confidence intervals from the test-set predictions.

**Interpretation:** SHAP values (TreeExplainer for RF/XGBoost, LinearExplainer for the linear model) computed on each model's test set using the already-fitted final model.

**Data note:** until the real train/test files land via GitHub, `_load_or_synthesize` derives all 4 from today's raw synthetic CSV (80/20 split, in-memory only, nothing written to disk) purely so this notebook can be exercised end-to-end — a no-op now that the real files exist at the configured paths.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    precision_recall_curve,
    brier_score_loss,
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier

import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

import shap
import joblib

In [ ]:
# --- config -----------------------------------------------------------------
# Tune these down for a fast smoke test, up for a production run.
RANDOM_STATE = 42
DIABETES_THRESHOLD = 6.5

N_INNER_FOLDS = 5
N_OPTUNA_TRIALS = 20
N_BOOTSTRAP = 500

MODEL_NAMES = ["logreg_elasticnet", "random_forest", "xgboost"]

# See the title cell: literal *_T1-only scope was the explicit decision, but
# family history / smoking / NSES are meaningful baseline risk factors excluded
# by that choice. Flip this to True to add them back in.
INCLUDE_STATIC_BASELINE_FEATURES = False

# Model A (T1 -> T2) and Model B (T1 -> T3) each get their own pre-split train/test files.
TRAIN_PATH_A = "../data/processed/df_trainA.xlsx"
TEST_PATH_A = "../data/processed/df_testA.xlsx"
TRAIN_PATH_B = "../data/processed/df_trainB.xlsx"
TEST_PATH_B = "../data/processed/df_testB.xlsx"

# Same directories as T1_T2 and T1_T3 model.ipynb (the with-spline version) on purpose - every
# filename below is tagged with RUN_TAG ("no_spline_run<N>"), so the two notebooks' outputs never
# collide even though they share MODELS_DIR/EXPORT_DIR.
# Not gitignored automatically - decide separately whether this should be committed.
MODELS_DIR = Path("../models")
EXPORT_DIR = Path("../outputs/summary_export")
FIGURES_DIR = EXPORT_DIR / "figures"
TABLES_DIR = EXPORT_DIR / "tables"

RUN_LABEL = "no_spline"


def _next_series_number(*directories, label=RUN_LABEL):
    """Scan the given directories for files already tagged '{label}_run<N>' and return the next
    unused N, so successive runs of this notebook get their own numbered files instead of silently
    overwriting the previous run's exports."""
    pattern = re.compile(rf"{re.escape(label)}_run(\d+)")
    numbers = []
    for d in directories:
        d = Path(d)
        if d.exists():
            for p in d.rglob("*"):
                m = pattern.search(p.name)
                if m:
                    numbers.append(int(m.group(1)))
    return max(numbers, default=0) + 1


RUN_SERIES_NUMBER = _next_series_number(MODELS_DIR, EXPORT_DIR)
RUN_TAG = f"{RUN_LABEL}_run{RUN_SERIES_NUMBER}"
print(f"This run is tagged: {RUN_TAG}")

## Load data

Each model gets its own pre-split train/test files (`TRAIN_PATH_A`/`TEST_PATH_A` for Model A, `TRAIN_PATH_B`/`TEST_PATH_B` for Model B) — no in-notebook train/test split happens anymore. `_read_table` dispatches on file extension (`.xlsx`/`.xls` -> `read_excel`, else `read_csv`) — the real files delivered so far are `.xlsx`, which `read_csv` can't parse (it's a binary format, not text, hence the `UnicodeDecodeError` if you pass an Excel file to it).

**TEMPORARY (testing only).** If the configured paths don't exist yet, `_load_or_synthesize` falls back to an in-memory 80/20 split of today's raw synthetic CSV so this notebook can still be exercised end-to-end. Once real files exist at the configured paths (as they now do), this fallback never triggers.

In [ ]:
import os


def _read_table(path):
    """Dispatch on file extension: Excel files need read_excel, not read_csv (which would try to
    decode the binary .xlsx format as UTF-8 text and raise a UnicodeDecodeError)."""
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    return pd.read_csv(path)


def _light_impute_for_testing(df):
    """Median/mode fallback so today's raw file can exercise this notebook. Not part of the real pipeline."""
    df = df.copy()
    for col in df.columns:
        if df[col].isna().any():
            if pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].fillna(df[col].median())
            else:
                mode = df[col].mode(dropna=True)
                if len(mode):
                    df[col] = df[col].fillna(mode.iloc[0])
    return df


def _load_or_synthesize(train_path, test_path, random_state=RANDOM_STATE):
    """Load the real train/test files if present; otherwise synthesize an 80/20 split from the raw
    synthetic file in-memory (nothing written to disk) purely so this notebook can be exercised."""
    if os.path.exists(train_path) and os.path.exists(test_path):
        return _read_table(train_path), _read_table(test_path)

    raw_path = "../data/raw/exquairo_ai_bootcamp_synth_dataset.csv"
    raw = pd.read_csv(raw_path, sep=";")
    raw = _light_impute_for_testing(raw)
    shuffled = raw.sample(frac=1.0, random_state=random_state)
    split_point = int(len(shuffled) * 0.8)
    train = shuffled.iloc[:split_point].reset_index(drop=True)
    test = shuffled.iloc[split_point:].reset_index(drop=True)
    return train, test


def ensure_diabetes_flags(df, threshold=DIABETES_THRESHOLD):
    """Derive DIABETES_T1/T2/T3 from HBAC_T1/T2/T3 > threshold if not already present. The real
    preprocessed files already include these as the canonical outcome/exclusion flags (confirmed to
    exactly match HBAC_* > 6.5 today, but treated as the source of truth in case that definition is
    ever refined upstream, e.g. to also account for diabetes medication)."""
    df = df.copy()
    for t in ["T1", "T2", "T3"]:
        flag_col, hbac_col = f"DIABETES_{t}", f"HBAC_{t}"
        if flag_col not in df.columns and hbac_col in df.columns:
            df[flag_col] = (df[hbac_col] > threshold).astype(int)
    return df


train_a, test_a = _load_or_synthesize(TRAIN_PATH_A, TEST_PATH_A)
train_b, test_b = _load_or_synthesize(TRAIN_PATH_B, TEST_PATH_B)

train_a, test_a, train_b, test_b = (ensure_diabetes_flags(df) for df in (train_a, test_a, train_b, test_b))

for name, df in {"train_a": train_a, "test_a": test_a, "train_b": train_b, "test_b": test_b}.items():
    print(f"{name}: shape={df.shape}, missing={df.isna().sum().sum()}")

## Cohort construction

Same exclusion logic applied independently to each of the 4 files: drop missing `HBAC_T1`, exclude prevalent diabetes at baseline (via `DIABETES_T1`), drop rows missing the target flag. `build_cohort` never reads a `T2` column when `target_suffix="T3"`, so Model B genuinely never touches T2 data. In the real files this exclusion is **not** a no-op — 197/13096 train-A rows still have `DIABETES_T1 == 1` (prevalent diabetes wasn't pre-excluded upstream), so this step is doing real work.

In [ ]:
def build_cohort(df, target_suffix):
    """Exclusion funnel + binary outcome for a T1-features -> DIABETES_{target_suffix} model.
    target_suffix: 'T2' or 'T3'. Uses the canonical DIABETES_T1/DIABETES_{target_suffix} flags
    (see ensure_diabetes_flags) rather than re-deriving from HBAC_* itself."""
    funnel = {"total": len(df)}

    cohort = df.dropna(subset=["HBAC_T1"])
    funnel["has HBAC_T1"] = len(cohort)

    cohort = cohort[cohort["DIABETES_T1"] == 0]
    funnel["no diabetes at T1"] = len(cohort)

    target_flag_col = f"DIABETES_{target_suffix}"
    cohort = cohort.dropna(subset=[target_flag_col])
    funnel[f"has {target_flag_col}"] = len(cohort)

    cohort = cohort.copy()
    outcome_col = f"{target_flag_col}_OUTCOME"
    cohort[outcome_col] = cohort[target_flag_col].astype(int)
    return cohort, funnel, outcome_col


cohort_a_train, funnel_a_train, outcome_col_a = build_cohort(train_a, "T2")
cohort_a_test, funnel_a_test, _ = build_cohort(test_a, "T2")
cohort_b_train, funnel_b_train, outcome_col_b = build_cohort(train_b, "T3")
cohort_b_test, funnel_b_test, _ = build_cohort(test_b, "T3")

for name, (cohort, funnel, outcome_col) in {
    "Model A / train (T1 -> T2)": (cohort_a_train, funnel_a_train, outcome_col_a),
    "Model A / test  (T1 -> T2)": (cohort_a_test, funnel_a_test, outcome_col_a),
    "Model B / train (T1 -> T3)": (cohort_b_train, funnel_b_train, outcome_col_b),
    "Model B / test  (T1 -> T3)": (cohort_b_test, funnel_b_test, outcome_col_b),
}.items():
    print(name)
    for step, n in funnel.items():
        print(f"  {step:20s} n={n:6d}")
    print(f"  prevalence: {cohort[outcome_col].mean():.2%} ({cohort[outcome_col].sum()} / {len(cohort)})")
    print()

## Feature selection

`*_T1` scope, including `*_T1_MISSING` missingness-indicator companions (present in the real files, e.g. `HBF_T1_MISSING`) — these end in `_MISSING`, not literally `_T1`, so a naive `endswith("_T1")` check silently drops them; fixed via a regex instead. `DIABETES_T1` is explicitly excluded: since the cohort filter above requires `DIABETES_T1 == 0`, it's *always* 0 within the cohort by construction — a genuinely zero-variance column, which would make `StandardScaler` divide by zero (0/0 -> `NaN`) for the logistic regression model. A generic zero-variance safety net (fit on train, applied to both train and test) catches this and anything else like it.

**Train/test column asymmetry:** `_MISSING` companion columns are only generated for a variable where that split actually had a missing value — so e.g. `HBF_T1_MISSING` can exist in the train file but not in test, if test happened to have zero missingness there. Computing each side's feature list independently would then try to select a column from test that doesn't exist. `get_shared_t1_features` handles this: for a `_MISSING` column present on only one side, it's kept and filled with `0` on the side that lacks it (absence of the column *means* no missingness there, so `0` is the correct value, not a guess); any other column present on only one side is dropped from both, since there's no safe default to assume.

In [ ]:
import re

# Columns that are trivially redundant with the cohort exclusion itself, not general-purpose exclusions.
STRUCTURALLY_CONSTANT_COLUMNS = {"DIABETES_T1"}


def get_t1_features(df, include_static=INCLUDE_STATIC_BASELINE_FEATURES):
    t1_pattern = re.compile(r"_T1(_MISSING)?$")
    t1_cols = [c for c in df.columns if t1_pattern.search(c) and c not in STRUCTURALLY_CONSTANT_COLUMNS]
    if include_static:
        static_cols = [c for c in df.columns if not re.search(r"_T[123](_MISSING)?$", c) and c != "ZIP_CODE"]
        return t1_cols + static_cols
    return t1_cols


def get_shared_t1_features(train_cohort, test_cohort, include_static=INCLUDE_STATIC_BASELINE_FEATURES):
    """Feature columns from either train or test's T1 scope, reconciled so both sides can actually
    provide every column. A '*_MISSING' companion present on only one side means that split had zero
    missingness for the underlying variable - kept, and filled with 0 on the side that lacks it (see
    build_model_dataset). Any other column present on only one side is dropped from both, since there's
    no safe default to assume for it."""
    train_cols = set(get_t1_features(train_cohort, include_static))
    test_cols = set(get_t1_features(test_cohort, include_static))

    shared, dropped = [], []
    for c in sorted(train_cols | test_cols):
        if c in train_cols and c in test_cols:
            shared.append(c)
        elif c.endswith("_MISSING"):
            shared.append(c)
        else:
            dropped.append(c)
    if dropped:
        print(f"  dropped (present in only train or only test, no safe default): {dropped}")
    return shared


def build_model_dataset(cohort, outcome_col, feature_cols):
    cohort = cohort.copy()
    for c in feature_cols:
        if c not in cohort.columns:
            cohort[c] = 0  # only '*_MISSING' columns can reach here (see get_shared_t1_features)
    X = cohort[feature_cols].copy()
    y = cohort[outcome_col].copy()
    return X, y


def drop_zero_variance_features(X_train, *other_frames, verbose=True):
    """Drop any feature that's constant in the training data - breaks StandardScaler (div by zero)
    and carries no signal anyway. The decision is made on TRAIN only; the same columns are dropped
    from every frame passed in `other_frames` (e.g. the matching test set) to keep columns aligned."""
    zero_var_cols = X_train.columns[X_train.nunique(dropna=False) <= 1].tolist()
    if zero_var_cols and verbose:
        print(f"  dropping zero-variance features (constant in train): {zero_var_cols}")
    kept_cols = [c for c in X_train.columns if c not in zero_var_cols]
    return (X_train[kept_cols],) + tuple(frame[kept_cols] for frame in other_frames)


print("Model A feature reconciliation:")
feature_cols_a = get_shared_t1_features(cohort_a_train, cohort_a_test)
print("Model B feature reconciliation:")
feature_cols_b = get_shared_t1_features(cohort_b_train, cohort_b_test)

X_a_train, y_a_train = build_model_dataset(cohort_a_train, outcome_col_a, feature_cols_a)
X_a_test, y_a_test = build_model_dataset(cohort_a_test, outcome_col_a, feature_cols_a)
X_b_train, y_b_train = build_model_dataset(cohort_b_train, outcome_col_b, feature_cols_b)
X_b_test, y_b_test = build_model_dataset(cohort_b_test, outcome_col_b, feature_cols_b)

print("Model A:")
X_a_train, X_a_test = drop_zero_variance_features(X_a_train, X_a_test)
print("Model B:")
X_b_train, X_b_test = drop_zero_variance_features(X_b_train, X_b_test)

for name, (X, y) in {
    "Model A / train": (X_a_train, y_a_train),
    "Model A / test": (X_a_test, y_a_test),
    "Model B / train": (X_b_train, y_b_train),
    "Model B / test": (X_b_test, y_b_test),
}.items():
    assert not any(c.endswith(("_T2", "_T3")) for c in X.columns), f"{name}: must not include T2/T3 data"
    assert "HBAC_T1" in X.columns, f"{name}: HBAC_T1 must be present as a continuous feature"
    assert set(y.unique()) <= {0, 1}, f"{name}: outcome must be binary"
    print(f"{name}: X={X.shape}, prevalence={y.mean():.2%}, dtypes={X.dtypes.value_counts().to_dict()}")

# Model A and B must use identical column sets (same T1 feature definition, different target/cohort)
assert list(X_a_train.columns) == list(X_b_train.columns), "Model A and Model B feature sets diverged"
print("No T2/T3 columns present in any feature matrix.")

## Modeling pipeline

Every model is `[preprocessor -> SMOTE -> classifier]`. The preprocessor is a `ColumnTransformer` built from the actual dtypes of whatever `X` it's given: numeric columns are scaled (logistic regression) or passed through as-is (tree models), and any non-numeric (object/category) column is one-hot encoded (`handle_unknown="ignore"` so a category unseen in training doesn't crash prediction). This means the pipeline auto-adapts if the real dataset turns out to have string/categorical `*_T1` columns instead of everything being pre-encoded as 0/1 — today's data happens to be all-numeric, so the categorical branch is present but untested against real string data until that arrives.

`imblearn` pipelines automatically skip the resampler at predict/transform time, so SMOTE only ever touches training folds. `k_neighbors` for SMOTE is capped by the minority class count in whatever fold it's fit on, since inner folds can have very few positives given the low prevalence.

In [ ]:
def safe_smote(y, random_state=RANDOM_STATE):
    minority_count = pd.Series(y).value_counts().min()
    k_neighbors = max(1, min(5, minority_count - 1))
    return SMOTE(random_state=random_state, k_neighbors=k_neighbors)


def build_preprocessor(X, model_name):
    """ColumnTransformer built from X's actual dtypes: numeric cols get scaled (LR) or passed through
    (tree models), any non-numeric col gets one-hot encoded. Detected fresh each call so it always
    matches whatever columns/dtypes the caller's X actually has."""
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    numeric_step = StandardScaler() if model_name == "logreg_elasticnet" else "passthrough"
    transformers = [("num", numeric_step, numeric_cols)]

    if categorical_cols:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols))
    return ColumnTransformer(transformers, remainder="drop")


def make_pipeline(model_name, X_for_columns, y_for_smote, random_state=RANDOM_STATE):
    preprocessor = build_preprocessor(X_for_columns, model_name)
    smote = safe_smote(y_for_smote, random_state)
    if model_name == "logreg_elasticnet":
        clf = LogisticRegression(penalty="elasticnet", solver="saga", max_iter=5000, random_state=random_state)
    elif model_name == "random_forest":
        clf = RandomForestClassifier(random_state=random_state, n_jobs=-1)
    elif model_name == "xgboost":
        clf = XGBClassifier(random_state=random_state, eval_metric="logloss", n_jobs=-1)
    else:
        raise ValueError(f"unknown model_name: {model_name}")
    return ImbPipeline([("preprocessor", preprocessor), ("smote", smote), ("clf", clf)])


def suggest_params(trial, model_name):
    if model_name == "logreg_elasticnet":
        return {
            "clf__C": trial.suggest_float("C", 1e-3, 1e2, log=True),
            "clf__l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
        }
    elif model_name == "random_forest":
        return {
            "clf__n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "clf__max_depth": trial.suggest_int("max_depth", 2, 20),
            "clf__min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            "clf__max_features": trial.suggest_float("max_features", 0.3, 1.0),
        }
    elif model_name == "xgboost":
        return {
            "clf__n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "clf__max_depth": trial.suggest_int("max_depth", 2, 10),
            "clf__learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "clf__subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "clf__colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "clf__reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        }
    else:
        raise ValueError(f"unknown model_name: {model_name}")

In [ ]:
def best_f1_threshold(y_true, y_proba):
    """Threshold that maximizes F1 on the given predictions."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    if len(thresholds) == 0:
        return 0.5
    f1s = 2 * precisions * recalls / (precisions + recalls + 1e-12)
    f1s = f1s[:-1]  # last precision/recall point has no corresponding threshold
    return float(thresholds[np.argmax(f1s)])


def select_threshold(X_train, y_train, model_name, best_params, n_inner_folds=N_INNER_FOLDS, random_state=RANDOM_STATE):
    """F1-optimal threshold from inner-CV out-of-fold predictions on X_train only (never touches any test set)."""
    inner_cv = StratifiedKFold(n_splits=n_inner_folds, shuffle=True, random_state=random_state)
    inner_oof_proba = np.zeros(len(y_train))
    for tr_idx, val_idx in inner_cv.split(X_train, y_train):
        fold_pipe = make_pipeline(model_name, X_train.iloc[tr_idx], y_train.iloc[tr_idx], random_state)
        fold_pipe.set_params(**best_params)
        fold_pipe.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        inner_oof_proba[val_idx] = fold_pipe.predict_proba(X_train.iloc[val_idx])[:, 1]
    return best_f1_threshold(y_train.values, inner_oof_proba)


def make_objective(X_train, y_train, model_name, n_inner_folds=N_INNER_FOLDS, random_state=RANDOM_STATE):
    """Inner-CV objective (mean AUPRC), reporting per-fold progress so Optuna can prune bad trials early."""

    def objective(trial):
        params = suggest_params(trial, model_name)
        inner_cv = StratifiedKFold(n_splits=n_inner_folds, shuffle=True, random_state=random_state)
        fold_scores = []
        for fold_i, (tr_idx, val_idx) in enumerate(inner_cv.split(X_train, y_train)):
            X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

            pipe = make_pipeline(model_name, X_tr, y_tr, random_state)
            pipe.set_params(**params)
            pipe.fit(X_tr, y_tr)
            proba = pipe.predict_proba(X_val)[:, 1]
            fold_scores.append(average_precision_score(y_val, proba))

            trial.report(float(np.mean(fold_scores)), fold_i)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(fold_scores))

    return objective

In [ ]:
def bootstrap_metrics(y_true, y_proba, threshold, n_bootstrap=N_BOOTSTRAP, random_state=RANDOM_STATE):
    """Point estimates + 95% bootstrap CIs for AUROC, AUPRC, precision, recall on pooled predictions."""
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    n = len(y_true)

    boot = {"auroc": [], "auprc": [], "precision": [], "recall": []}
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        yt, yp = y_true[idx], y_proba[idx]
        if len(np.unique(yt)) < 2:
            continue
        preds = (yp >= threshold).astype(int)
        boot["auroc"].append(roc_auc_score(yt, yp))
        boot["auprc"].append(average_precision_score(yt, yp))
        boot["precision"].append(precision_score(yt, preds, zero_division=0))
        boot["recall"].append(recall_score(yt, preds, zero_division=0))

    preds = (y_proba >= threshold).astype(int)
    point = {
        "auroc": roc_auc_score(y_true, y_proba),
        "auprc": average_precision_score(y_true, y_proba),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
    }
    ci = {m: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) for m, v in boot.items()}
    return point, ci

## Assemble per-model datasets

In [ ]:
DATASETS = {
    "T1_to_T2": {"train": (X_a_train, y_a_train), "test": (X_a_test, y_a_test)},
    "T1_to_T3": {"train": (X_b_train, y_b_train), "test": (X_b_test, y_b_test)},
}

## Final model + external test evaluation

For each model/algorithm: an Optuna search (5-fold inner CV, AUPRC objective, pruned) + refit on the **full train file**, an F1-optimal threshold selected via inner CV on train only, then evaluation on both the train file itself (in-sample, at the same threshold) and the held-out test file. Nothing in the tuning or fitting process ever sees the test file — only the already-fitted final model is used to score it.

In [ ]:
def fit_final_model(X, y, model_name, n_trials=N_OPTUNA_TRIALS, random_state=RANDOM_STATE):
    """Optuna search + refit on the given data. Returns (fitted_pipe, best_params)."""
    sampler = optuna.samplers.TPESampler(seed=random_state)
    pruner = optuna.pruners.MedianPruner(n_warmup_steps=1)
    study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
    study.optimize(make_objective(X, y, model_name, random_state=random_state), n_trials=n_trials)

    best_params = {f"clf__{k}": v for k, v in study.best_params.items()}
    pipe = make_pipeline(model_name, X, y, random_state)
    pipe.set_params(**best_params)
    pipe.fit(X, y)
    return pipe, best_params


def evaluate_final_model(X_train, y_train, X_test, y_test, model_name, n_trials=N_OPTUNA_TRIALS, random_state=RANDOM_STATE):
    pipe, best_params = fit_final_model(X_train, y_train, model_name, n_trials=n_trials, random_state=random_state)
    threshold = select_threshold(X_train, y_train, model_name, best_params, random_state=random_state)

    # In-sample (train) predictions: the model scoring the same data it was fit on - optimistic by
    # construction, kept only as a reference point for the train-vs-test comparison below, never as
    # a standalone performance estimate.
    train_proba = pipe.predict_proba(X_train)[:, 1]
    test_proba = pipe.predict_proba(X_test)[:, 1]

    return {
        "model_name": model_name,
        "pipe": pipe,
        "best_params": best_params,
        "y_train_true": y_train.values,
        "y_train_proba": train_proba,
        "y_true": y_test.values,
        "y_proba": test_proba,
        "threshold": threshold,
    }


final_results = {}
for outcome_name, splits in DATASETS.items():
    X_train, y_train = splits["train"]
    X_test, y_test = splits["test"]
    for model_name in MODEL_NAMES:
        print(f"=== [final, external test] {outcome_name} / {model_name} ===")
        final_results[(outcome_name, model_name)] = evaluate_final_model(X_train, y_train, X_test, y_test, model_name)

In [ ]:
def build_performance_summary(results, true_key, proba_key):
    """One row per (outcome, model): % abnormal (positive-class prevalence) + bootstrap metrics at
    the model's selected threshold, evaluated on whichever (true_key, proba_key) pair is passed."""
    rows = []
    for (outcome_name, model_name), res in results.items():
        y_true, y_proba, threshold = res[true_key], res[proba_key], res["threshold"]
        point, ci = bootstrap_metrics(y_true, y_proba, threshold)
        rows.append(
            {
                "outcome": outcome_name,
                "model": model_name,
                "threshold": round(threshold, 3),
                "% Abnormal": f"{100 * np.mean(y_true):.2f}%",
                "AUROC": f"{point['auroc']:.3f} [{ci['auroc'][0]:.3f}, {ci['auroc'][1]:.3f}]",
                "AUPRC": f"{point['auprc']:.3f} [{ci['auprc'][0]:.3f}, {ci['auprc'][1]:.3f}]",
                "Precision": f"{point['precision']:.3f} [{ci['precision'][0]:.3f}, {ci['precision'][1]:.3f}]",
                "Recall": f"{point['recall']:.3f} [{ci['recall'][0]:.3f}, {ci['recall'][1]:.3f}]",
            }
        )
    return pd.DataFrame(rows)


train_summary_df = build_performance_summary(final_results, "y_train_true", "y_train_proba")
test_summary_df = build_performance_summary(final_results, "y_true", "y_proba")

print("Train (in-sample, optimistic) performance at the selected threshold:")
display(train_summary_df)
print()
print("Test (external, unbiased) performance at the selected threshold:")
display(test_summary_df)

## Validation analyses

**Calibration + Brier score** on the final model's predictions on the real test file — the genuine unseen-data check on whether predicted probabilities are trustworthy.

In [ ]:
def plot_calibration(final_results, outcome_name):
    fig, ax = plt.subplots(figsize=(6, 6))
    for (name, model_name), res in final_results.items():
        if name != outcome_name:
            continue
        frac_pos, mean_pred = calibration_curve(res["y_true"], res["y_proba"], n_bins=10, strategy="quantile")
        brier = brier_score_loss(res["y_true"], res["y_proba"])
        ax.plot(mean_pred, frac_pos, marker="o", label=f"{model_name} (Brier={brier:.3f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="perfectly calibrated")
    ax.set_xlabel("mean predicted probability")
    ax.set_ylabel("fraction of positives")
    ax.set_title(f"Calibration on external test - {outcome_name}")
    ax.legend()
    fig.tight_layout()
    plt.show()


for outcome_name in DATASETS:
    plot_calibration(final_results, outcome_name)

## SHAP feature importance

Reuses the final models already fitted on each train file (`final_results[...]["pipe"]`) — no redundant refitting. SHAP values are computed on each model's **test** file, i.e. genuine held-out data. `TreeExplainer` (RF/XGBoost) or `LinearExplainer` (logistic regression) runs on the preprocessed features; feature names come from `preprocessor.get_feature_names_out()` rather than the original `X.columns`, since one-hot encoding a categorical column (if the real dataset has any) would change the column count and break a naive 1:1 name mapping.

Alongside each beeswarm plot, a numeric ranking table (mean absolute SHAP value per feature) is printed and stored in `shap_results`, so the underlying values are available for reporting or further analysis, not just the visualization.

In [ ]:
def compute_shap(pipe, X, model_name):
    preprocessor = pipe.named_steps["preprocessor"]
    X_transformed = preprocessor.transform(X)
    # ColumnTransformer prefixes names as "num__COL" / "cat__COL_value"; drop the "num__" prefix for
    # readability, keep "cat__" since one-hot levels need it to stay distinguishable.
    feature_names = [name.removeprefix("num__") for name in preprocessor.get_feature_names_out()]
    X_transformed = pd.DataFrame(X_transformed, columns=feature_names, index=X.index)

    clf = pipe.named_steps["clf"]
    if model_name in ("random_forest", "xgboost"):
        explainer = shap.TreeExplainer(clf)
    else:
        explainer = shap.LinearExplainer(clf, X_transformed)

    shap_values = np.asarray(explainer.shap_values(X_transformed))
    base_value = np.asarray(explainer.expected_value)

    if shap_values.ndim == 3:  # some TreeExplainer versions return (n_samples, n_features, n_classes)
        shap_values = shap_values[:, :, 1]
    base_value = base_value.reshape(-1)
    base_value = float(base_value[1] if base_value.size > 1 else base_value[0])  # positive-class base rate

    return shap_values, X_transformed, base_value


def summarize_shap_importance(shap_values, X_transformed, top_n=15):
    """Mean absolute SHAP value per feature, descending - the numeric counterpart to the beeswarm plot."""
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    importance = pd.Series(mean_abs_shap, index=X_transformed.columns, name="mean_abs_shap")
    return importance.sort_values(ascending=False).head(top_n).to_frame()

In [ ]:
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

shap_results = {}
for outcome_name, splits in DATASETS.items():
    X_test, _ = splits["test"]
    for model_name in MODEL_NAMES:
        pipe = final_results[(outcome_name, model_name)]["pipe"]
        shap_values, X_transformed, base_value = compute_shap(pipe, X_test, model_name)
        importance_table = summarize_shap_importance(shap_values, X_transformed)
        shap_results[(outcome_name, model_name)] = {
            "shap_values": shap_values,
            "X_transformed": X_transformed,
            "base_value": base_value,
            "importance": importance_table,
        }

        plt.figure()
        shap.summary_plot(shap_values, X_transformed, show=False, max_display=15)
        plt.title(f"{outcome_name} - {model_name} (test set)")
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f"shap_beeswarm_{outcome_name}_{model_name}_{RUN_TAG}.png", dpi=150, bbox_inches="tight")
        plt.show()

        print(f"Top SHAP feature importances (mean |SHAP|) - {outcome_name} / {model_name}:")
        display(importance_table)
        print()

## Demographics table

Baseline (T1) characteristics of each model's **final analytic population** (train + test combined, after all cohort exclusions). Continuous variables are summarized as mean (SD); binary variables (coded 0/1) as n (%) of the coded-1 group. Variables are auto-detected as binary vs. continuous by cardinality (`<= 10` unique values -> binary), same convention used elsewhere in this notebook. `*_MISSING` indicator columns and the structurally-constant `DIABETES_T1` are excluded from the table itself, since they're data-quality artifacts rather than demographic/clinical characteristics.

In [ ]:
DEMOGRAPHIC_CARDINALITY_THRESHOLD = 10


def is_binary_demographic(series, threshold=DEMOGRAPHIC_CARDINALITY_THRESHOLD):
    return series.nunique(dropna=False) <= threshold


def summarize_demographic_variable(series):
    if is_binary_demographic(series):
        n_positive = int((series == 1).sum())
        pct = 100 * series.mean()
        return f"{n_positive} ({pct:.1f}%)"
    return f"{series.mean():.2f} ({series.std():.2f})"


def get_demographic_variables(df):
    exclude = {"DIABETES_T1"}
    return [
        c for c in df.columns
        if (c.endswith("_T1") or c == "GENDER") and not c.endswith("_MISSING") and c not in exclude
    ]


def build_demographics_table(cohorts):
    """cohorts: {label: dataframe}. One row per baseline variable, one column per cohort."""
    demo_vars = get_demographic_variables(next(iter(cohorts.values())))

    rows = [{"Variable": "N", **{label: len(df) for label, df in cohorts.items()}}]
    for var in demo_vars:
        row = {"Variable": var}
        for label, df in cohorts.items():
            row[label] = summarize_demographic_variable(df[var])
        rows.append(row)

    return pd.DataFrame(rows).set_index("Variable")


cohort_a_full = pd.concat([cohort_a_train, cohort_a_test], ignore_index=True)
cohort_b_full = pd.concat([cohort_b_train, cohort_b_test], ignore_index=True)

DEMOGRAPHIC_COHORTS = {"Model A (T1 -> T2)": cohort_a_full, "Model B (T1 -> T3)": cohort_b_full}

demographics_table = build_demographics_table(DEMOGRAPHIC_COHORTS)
print("Continuous: mean (SD). Binary: n (%) of coded 1.")
demographics_table

In [ ]:
def plot_demographics(cohorts, continuous_vars, binary_vars):
    """Boxplots for continuous variables + grouped bar charts for binary variables, comparing cohorts."""
    n_panels = len(continuous_vars) + 2  # +1 for the headline binary var, +1 for the rest grouped
    n_cols = 3
    n_rows = -(-n_panels // n_cols)  # ceil
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4.5 * n_rows))
    axes = np.atleast_1d(axes).ravel()

    labels = list(cohorts.keys())

    for ax, var in zip(axes, continuous_vars):
        data = [df[var].dropna() for df in cohorts.values()]
        ax.boxplot(data, tick_labels=labels)
        ax.set_title(var)
        ax.tick_params(axis="x", rotation=15)

    headline_var, rest_vars = binary_vars[0], binary_vars[1:]

    ax = axes[len(continuous_vars)]
    pct_positive = [100 * df[headline_var].mean() for df in cohorts.values()]
    ax.bar(labels, pct_positive, color="#4C72B0")
    ax.set_title(f"{headline_var} (% coded 1)")
    ax.set_ylabel("%")
    ax.set_ylim(0, 100)
    ax.tick_params(axis="x", rotation=15)

    ax = axes[len(continuous_vars) + 1]
    x = np.arange(len(rest_vars))
    width = 0.8 / len(cohorts)
    for i, (label, df) in enumerate(cohorts.items()):
        pct = [100 * df[v].mean() for v in rest_vars]
        ax.bar(x + i * width, pct, width, label=label)
    ax.set_xticks(x + width * (len(cohorts) - 1) / 2)
    ax.set_xticklabels(rest_vars, rotation=30, ha="right")
    ax.set_ylabel("%")
    ax.set_title("Other baseline binary characteristics")
    ax.legend(fontsize=8)

    for ax in axes[len(continuous_vars) + 2:]:
        ax.axis("off")

    fig.suptitle("Baseline (T1) characteristics by model cohort", fontsize=14)
    fig.tight_layout()
    plt.show()


demo_vars = get_demographic_variables(cohort_a_full)
continuous_vars = [v for v in demo_vars if not is_binary_demographic(cohort_a_full[v])]
binary_vars = [v for v in demo_vars if is_binary_demographic(cohort_a_full[v])]
# GENDER first if present, so it becomes the headline binary panel
binary_vars = sorted(binary_vars, key=lambda v: v != "GENDER")

plot_demographics(DEMOGRAPHIC_COHORTS, continuous_vars, binary_vars)

## Best model selection and export

**Illustrate:** a grouped bar chart per model (A, B) comparing all 3 algorithms across AUROC, AUPRC, precision, and recall on the external test set, with 95% bootstrap error bars — the same numbers as the summary table above, but visually easier to compare across algorithms at a glance.

**Select:** for each model, the algorithm with the highest test-set AUPRC is chosen as "best" — AUPRC is the metric already used to drive hyperparameter tuning, and is more informative than AUROC given the low outcome prevalence (~1-2.7%).

**Export:** the winning fitted `Pipeline` (preprocessing + SMOTE + classifier, as one object) is saved with `joblib.dump` — anyone loading it just needs to supply the raw `*_T1` columns; the pipeline itself replays the scaling/encoding and feeds the classifier. A sidecar JSON captures everything needed to interpret or reproduce it: the decision threshold, tuned hyperparameters, expected feature columns, and test-set performance. Both land in `../models/`, which is gitignored (model binaries don't belong in version control).

**Naming:** every exported filename here is suffixed with `RUN_TAG` (`no_spline_run<N>`, e.g. `model_b_best_random_forest_no_spline_run1.joblib`) — `no_spline` distinguishes this notebook's output from `T1_T2 and T1_T3 model.ipynb` (the with-spline version, which shares the same `../models/` directory but writes untagged filenames), and `run<N>` is an auto-incrementing series number (`_next_series_number` in the config cell) so re-running this notebook doesn't overwrite its own previous exports either.

In [ ]:
def plot_model_comparison(final_results, outcome_name, metrics=("auroc", "auprc", "precision", "recall")):
    algo_results = {model_name: res for (name, model_name), res in final_results.items() if name == outcome_name}

    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(metrics))
    width = 0.8 / len(algo_results)

    for i, (model_name, res) in enumerate(algo_results.items()):
        point, ci = bootstrap_metrics(res["y_true"], res["y_proba"], res["threshold"])
        values = [point[m] for m in metrics]
        lower_err = [point[m] - ci[m][0] for m in metrics]
        upper_err = [ci[m][1] - point[m] for m in metrics]
        ax.bar(x + i * width, values, width, yerr=[lower_err, upper_err], capsize=3, label=model_name)

    ax.set_xticks(x + width * (len(algo_results) - 1) / 2)
    ax.set_xticklabels([m.upper() for m in metrics])
    ax.set_ylim(0, 1)
    ax.set_ylabel("score")
    ax.set_title(f"Algorithm comparison on external test - {outcome_name}")
    ax.legend()
    fig.tight_layout()
    plt.show()


for outcome_name in DATASETS:
    plot_model_comparison(final_results, outcome_name)

In [ ]:
BEST_MODEL_METRIC = "auprc"
OUTCOME_TO_MODEL_LABEL = {"T1_to_T2": "model_a", "T1_to_T3": "model_b"}
# MODELS_DIR and RUN_TAG are set in the config cell.


def select_best_model(final_results, outcome_name, metric=BEST_MODEL_METRIC):
    candidates = {model_name: res for (name, model_name), res in final_results.items() if name == outcome_name}
    scores = {}
    for model_name, res in candidates.items():
        point, _ = bootstrap_metrics(res["y_true"], res["y_proba"], res["threshold"])
        scores[model_name] = point[metric]
    best_model_name = max(scores, key=scores.get)
    return best_model_name, scores


def export_best_model(outcome_name, model_name, feature_columns):
    res = final_results[(outcome_name, model_name)]
    label = OUTCOME_TO_MODEL_LABEL[outcome_name]

    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    pipe_path = MODELS_DIR / f"{label}_best_{model_name}_{RUN_TAG}.joblib"
    meta_path = MODELS_DIR / f"{label}_best_{model_name}_{RUN_TAG}_metadata.json"

    joblib.dump(res["pipe"], pipe_path)

    point, ci = bootstrap_metrics(res["y_true"], res["y_proba"], res["threshold"])
    metadata = {
        "outcome": outcome_name,
        "algorithm": model_name,
        "run_tag": RUN_TAG,
        "threshold": res["threshold"],
        "best_params": res["best_params"],
        "feature_columns": list(feature_columns),
        "test_performance": {
            "point": point,
            "ci_95": {k: list(v) for k, v in ci.items()},
        },
    }
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2, default=str)

    print(f"exported: {pipe_path} , {meta_path}")
    return pipe_path, meta_path


best_models = {}
exported_paths = {}
for outcome_name in DATASETS:
    best_name, scores = select_best_model(final_results, outcome_name)
    best_models[outcome_name] = best_name
    print(f"{outcome_name}: best algorithm by {BEST_MODEL_METRIC.upper()} = {best_name}")
    print(f"  scores: {{ {', '.join(f'{k}: {v:.3f}' for k, v in scores.items())} }}")

    feature_columns = DATASETS[outcome_name]["train"][0].columns
    exported_paths[outcome_name] = export_best_model(outcome_name, best_name, feature_columns)

## SHAP decision plot for each best model

The beeswarm above shows population-level feature importance; a decision plot instead traces individual prediction paths — how each feature's contribution accumulates from the population base rate (`base_value`) up to the final predicted probability, one line per observation. It's useful for seeing whether predictions are driven by the same few dominant features across cases, or by different feature combinations for different people.

Only the winning algorithm per model (from `best_models` above) is plotted, reusing the already-computed `shap_results` — no recomputation. Limited to a random subset of `DECISION_PLOT_N_SAMPLES` test-set observations (fixed seed) to stay legible; a full test set of thousands of lines would be unreadable.

In [ ]:
DECISION_PLOT_N_SAMPLES = 50


def plot_shap_decision(outcome_name, model_name, n_samples=DECISION_PLOT_N_SAMPLES, random_state=RANDOM_STATE):
    res = shap_results[(outcome_name, model_name)]
    shap_values, X_transformed, base_value = res["shap_values"], res["X_transformed"], res["base_value"]

    rng = np.random.default_rng(random_state)
    n = len(X_transformed)
    idx = rng.choice(n, size=min(n_samples, n), replace=False)

    plt.figure(figsize=(9, 8))
    shap.decision_plot(
        base_value,
        shap_values[idx],
        X_transformed.iloc[idx],
        feature_display_range=slice(-1, -16, -1),  # top 15 features, same cap as the beeswarm
        show=False,
    )
    plt.title(f"{outcome_name} - best model ({model_name}): SHAP decision plot (n={len(idx)} test cases)")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"shap_decision_{outcome_name}_{model_name}_{RUN_TAG}.png", dpi=150, bbox_inches="tight")
    plt.show()


for outcome_name, model_name in best_models.items():
    plot_shap_decision(outcome_name, model_name)

## Export summary bundle

Everything from this run in one place: the SHAP beeswarm and decision plot figures already saved to `FIGURES_DIR` above, the train/test performance tables (all 3 algorithms × 2 models — `% Abnormal`, AUROC, AUPRC, precision, recall with bootstrap CIs), and the SHAP values themselves — both the raw per-observation array and the mean-|SHAP| importance ranking, for every model/algorithm combination, not just the winning ones.

Every table/figure filename is tagged with `RUN_TAG` (see the config cell), so this run's files sit alongside — not on top of — any previous run's, and never collide with `T1_T2 and T1_T3 model.ipynb`'s untagged output even though both write to `EXPORT_DIR`. The zip itself is named `summary_export_{RUN_TAG}.zip` (a sibling of `EXPORT_DIR`, not nested inside it) and bundles whatever is currently in `EXPORT_DIR` — which accumulates every tagged run, by design, so each zip is a complete snapshot as of that run.

Not gitignored automatically (see the config cell) — decide separately whether `../outputs/` should be committed or excluded, same as `../models/` above.

In [ ]:
import shutil

TABLES_DIR.mkdir(parents=True, exist_ok=True)

train_summary_df.to_csv(TABLES_DIR / f"train_performance_summary_{RUN_TAG}.csv", index=False)
test_summary_df.to_csv(TABLES_DIR / f"test_performance_summary_{RUN_TAG}.csv", index=False)

for (outcome_name, model_name), res in shap_results.items():
    shap_df = pd.DataFrame(res["shap_values"], columns=res["X_transformed"].columns)
    shap_df.to_csv(TABLES_DIR / f"shap_values_{outcome_name}_{model_name}_{RUN_TAG}.csv", index=False)
    res["importance"].to_csv(TABLES_DIR / f"shap_importance_{outcome_name}_{model_name}_{RUN_TAG}.csv")

zip_path = shutil.make_archive(str(EXPORT_DIR.parent / f"{EXPORT_DIR.name}_{RUN_TAG}"), "zip", root_dir=EXPORT_DIR)

print(f"exported bundle: {zip_path}")
print(f"  figures: {len(list(FIGURES_DIR.glob('*.png')))} PNGs")
print(f"  tables:  {len(list(TABLES_DIR.glob('*.csv')))} CSVs")